# LoRA & QLoRA — Code Companion (Unsloth + Llama 3.1 (8B))

This notebook accompanies **Topic: LoRA & QLoRA**.

This is the full, working training pipeline referenced throughout the mini-course:
fine-tuning **Llama 3.1 (8B)** on the **Alpaca** instruction dataset using **LoRA/QLoRA**
via **Unsloth**, following the structure of Unsloth's official
`Llama3.1 (8B) - Alpaca` Colab notebook.

> **Important:** this notebook needs a GPU (a free Google Colab T4 is enough) and
> internet access to download the base model and dataset. It will not run inside a
> CPU-only, offline sandbox. Every cell is complete, runnable code — open this in
> Colab, set the runtime to GPU, and run cells top to bottom.

**What you'll do:**
1. Install Unsloth and its dependencies.
2. Load Llama 3.1 (8B) in 4-bit (QLoRA).
3. Attach LoRA adapters, understanding every hyperparameter as you set it.
4. Prepare the Alpaca dataset with the prompt template + EOS token.
5. Train with TRL's `SFTTrainer`.
6. Run inference on your fine-tuned model.
7. Save and export the model in several formats.
8. Inspect the training loss curve.

## 0. Environment Check

Run this first. If `torch.cuda.is_available()` prints `False`, switch your runtime to a
GPU before continuing (in Colab: **Runtime → Change runtime type → GPU**).

In [ ]:
import torch
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1. Install Unsloth

One command installs Unsloth along with compatible, pinned versions of `transformers`,
`trl`, `peft`, `accelerate`, and `bitsandbytes` — the whole training stack.

In [ ]:
%%capture
!pip install unsloth
# If you hit a dependency conflict, Unsloth also publishes a "nightly" install command
# on its GitHub README that resolves the most common version clashes.

## 2. Load Llama 3.1 (8B) in 4-bit

`FastLanguageModel.from_pretrained` handles downloading the model, quantizing it to
4-bit (this is the "Q" in QLoRA), and loading a matching tokenizer, all in one call.

In [ ]:
from unsloth import FastLanguageModel

max_seq_length = 2048   # Llama 3.1 supports much longer, but this keeps training fast
dtype = None             # None = auto-detect the best dtype (bfloat16 on modern GPUs)
load_in_4bit = True      # quantize the frozen base model to 4-bit -> this is QLoRA

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

print(type(model).__name__, "loaded.")
print(type(tokenizer).__name__, "loaded.")

## 3. Attach LoRA Adapters

This wraps the model with small trainable low-rank matrices and freezes everything else.
Each argument below maps directly to a concept from the slides:

- **`r`** — the LoRA rank. Higher = more capacity, more trainable parameters. 16 is a
  common, well-tested default.
- **`target_modules`** — which weight matrices get an adapter. Here we target all the
  attention projections *and* the feed-forward matrices, for maximum coverage.
- **`lora_alpha`** — a scaling factor for the LoRA update; a common convention is to set
  it equal to `r`.
- **`lora_dropout`** — dropout on the LoRA path; `0` is specifically optimized in Unsloth's
  fused kernels, so it's the recommended default.
- **`use_gradient_checkpointing = "unsloth"`** — Unsloth's custom checkpointing, which
  trades a little compute for a large additional memory saving.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# Confirm how few parameters are actually trainable
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,}  ({trainable/total:.3%})")

## 4. Prepare the Alpaca Dataset

Same template and EOS-token logic covered in detail in the Data Preparation notebook,
applied here to the real dataset we're about to train on.

In [ ]:
from datasets import load_dataset

alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token   # MUST add this -- otherwise generation never stops

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input_, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input_, output) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

dataset = load_dataset("unsloth/alpaca-cleaned", split="train")
dataset = dataset.map(formatting_prompts_func, batched=True)

print(f"Loaded {len(dataset):,} examples")
print(dataset[0]["text"])

## 5. Train with TRL's `SFTTrainer`

`SFTTrainer` handles the training loop, batching, and checkpointing for you. This demo
run uses `max_steps` for a short, fast training run — for a full run, replace it with
`num_train_epochs` (e.g. `1` to `3`).

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,   # effective batch size = 2 * 4 = 8
        warmup_steps = 5,
        max_steps = 60,                    # short demo run; use num_train_epochs=1-3 for real training
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

In [ ]:
# Show current GPU memory before training, for reference
gpu_stats = torch.cuda.get_device_properties(0)
start_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}, max memory = {round(gpu_stats.total_memory / 1024**3, 3)} GB")
print(f"Memory reserved before training: {start_memory} GB")

trainer_stats = trainer.train()

In [ ]:
# Show memory and time stats after training
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
print(f"Peak reserved memory during training: {used_memory} GB")
print(f"Training time: {trainer_stats.metrics['train_runtime']:.1f} seconds")

## 6. Run Inference on the Fine-tuned Model

`FastLanguageModel.for_inference` switches the model into Unsloth's optimized inference
mode (roughly 2x faster generation).

In [ ]:
FastLanguageModel.for_inference(model)

inputs = tokenizer(
    [alpaca_prompt.format(
        "Continue the Fibonacci sequence.",
        "1, 1, 2, 3, 5, 8",
        "",   # leave the response blank -- the model fills this in
    )],
    return_tensors = "pt",
).to("cuda")

outputs = model.generate(**inputs, max_new_tokens=64, use_cache=True)
print(tokenizer.batch_decode(outputs, skip_special_tokens=True)[0])

## 7. Save & Export the Model

Unsloth supports several export paths depending on how you plan to deploy the model.

In [ ]:
# Save just the small LoRA adapter (a few MB) -- this is usually all you need
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")
print("Saved LoRA adapter to ./lora_model")

In [ ]:
# Optional: merge LoRA into the base weights and save a full 16-bit model
# model.save_pretrained_merged("model_16bit", tokenizer, save_method="merged_16bit")

# Optional: export directly to GGUF for llama.cpp / Ollama
# model.save_pretrained_gguf("model_gguf", tokenizer, quantization_method="q4_k_m")

# Optional: push the LoRA adapter straight to the Hugging Face Hub
# model.push_to_hub("your-username/llama-3.1-8b-alpaca-lora", tokenizer=tokenizer)

print("Uncomment whichever export path matches your deployment target.")

## 8. Reading the Training Loss

Plot the loss values Unsloth/TRL logged during training. A healthy run trends steadily
downward; a flat line or a spike to `NaN` both signal a problem worth investigating
(learning rate too low/high, or a data formatting bug).

In [ ]:
import matplotlib.pyplot as plt

log_history = trainer.state.log_history
steps = [entry["step"] for entry in log_history if "loss" in entry]
losses = [entry["loss"] for entry in log_history if "loss" in entry]

plt.figure(figsize=(7, 4))
plt.plot(steps, losses, marker="o", markersize=3)
plt.xlabel("Training step")
plt.ylabel("Loss")
plt.title("Training Loss Over Time")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Recap & Try It Yourself

You just walked the complete Llama 3.1 (8B) + Alpaca + Unsloth pipeline:
installation → 4-bit loading (QLoRA) → LoRA adapter setup → data formatting →
`SFTTrainer` → inference → export → reading the loss curve.

**Things to try:**
1. Change `r` from 16 to 8 or 32 and compare the trainable-parameter percentage printed
   in Section 3.
2. Replace `max_steps = 60` with `num_train_epochs = 1` for a full pass over the dataset
   (this will take considerably longer).
3. Swap the Alpaca dataset for your own instruction data, formatted the same way, and
   fine-tune on your own task.
4. After training, try a few of your own prompts in Section 6 and see how the responses
   compare to the base model's.